In [1]:
import torch
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle
import os
import csv

/opt/anaconda3/envs/MachLearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from plm_compare_progen2 import *
from protein_data import *

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

model_name = "hugohrban/progen2-medium"
model, tokenizer = initialize_progen2(model_name)

Using cpu device


In [6]:
def wild_type(mut_name, sequence):

    mut_list = mut_name.split(":")

    for x in mut_list:
        mut = x
        len_mut = len(mut)
        orig = mut[0]
        pos = int(mut[1:len_mut-1])-1
        new = mut[len_mut-1]

        if new==sequence[pos]:
            # print(f"Position {pos + 1} changed from {new} to {orig}.")
            wild_seq = sequence[:pos] + orig + sequence[pos+1:]
        else:
            return f"Amino acid {new} not in position {pos + 1}."
        
    return wild_seq
    

In [20]:
AA = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_idx = {aa: i for i, aa in enumerate(AA)}

directory = "/Users/johnhutchens/Desktop/Practicum/Data/zInput_Data/DMS_ProteinGym_substitutions"

pg_dict = dict()

for filename in os.listdir(directory):
    if filename.lower().endswith(".csv"):

        filename_list = filename.rsplit(".", 1)[0]
        parts = filename_list.split("_")

        if len(parts) >= 2:
            name = "_".join(parts[:2])

        filepath = os.path.join(directory, filename)

        with open(filepath, newline="", encoding="utf-8") as f:

            reader = csv.reader(f)
            next(reader, None)
            first_row = next(reader, None)
            wild = wild_type(first_row[0], first_row[1])

            n = len(wild)

            DMS = np.full((n, 20), np.nan)
            for row in reader:
                mut = row[0]
                #print(mut)
                if ":" not in mut:
                    index0 = int(mut[1:-1]) - 1
                    index1 = aa_to_idx[mut[-1]]
                    #print(index0, mut[-1], index1)
                    DMS[index0, index1] = row[2]
            
            pg_dict[name] = {'sequence': wild, 'DMS': DMS}
            
            

In [25]:
for key in list(pg_dict.keys()):

    print(pg_dict[key]['DMS'].shape,
    len(pg_dict[key]['sequence']) )

(44, 20) 44
(402, 20) 402
(553, 20) 553
(220, 20) 220
(346, 20) 346
(805, 20) 805
(757, 20) 757
(258, 20) 258
(128, 20) 128
(536, 20) 536
(248, 20) 248
(150, 20) 150
(286, 20) 286
(372, 20) 372
(577, 20) 577
(860, 20) 860
(66, 20) 66
(2016, 20) 2016
(61, 20) 61
(72, 20) 72
(54, 20) 54
(759, 20) 759
(72, 20) 72
(370, 20) 370
(67, 20) 67
(505, 20) 505
(198, 20) 198
(177, 20) 177
(770, 20) 770
(63, 20) 63
(393, 20) 393
(40, 20) 40
(364, 20) 364
(47, 20) 47
(159, 20) 159
(656, 20) 656
(635, 20) 635
(48, 20) 48
(934, 20) 934
(1278, 20) 1278
(852, 20) 852
(287, 20) 287
(55, 20) 55
(352, 20) 352
(55, 20) 55
(72, 20) 72
(3423, 20) 3423
(118, 20) 118
(448, 20) 448
(414, 20) 414
(61, 20) 61
(709, 20) 709
(66, 20) 66
(86, 20) 86
(465, 20) 465
(264, 20) 264
(589, 20) 589
(627, 20) 627
(70, 20) 70
(716, 20) 716
(211, 20) 211
(724, 20) 724
(861, 20) 861
(101, 20) 101
(93, 20) 93
(212, 20) 212
(71, 20) 71
(188, 20) 188
(281, 20) 281
(69, 20) 69
(490, 20) 490
(58, 20) 58
(565, 20) 565
(582, 20) 582
(7

In [35]:
keys = list(pg_dict.keys())

for i in range(len(keys)):
    key = keys[i]
    seq = pg_dict[key]['sequence']
    if len(seq) <= 1024:
        lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
        pg_dict[key]['log_probs'] = lp
        pg_dict[key]['ref_log_probs'] = rlp
        pg_dict[key]['llr_matrix'] = llr




In [36]:
pg_dict[keys[43]]

{'sequence': 'MDYQVSSPIYDINYYTSEPCQKINVKQIAARLLPPLYSLVFIFGFVGNMLVILILINCKRLKSMTDIYLLNLAISDLFFLLTVPFWAHYAAAQWDFGNTMCQLLTGLYFIGFFSGIFFIILLTIDRYLAVVHAVFALKARTVTFGVVTSVITWVVAVFASLPGIIFTRSQKEGLHYTCSSHFPYSQYQFWKNFQTLKIVILGLVLPLLVMVICYSGILKTLLRCRNEKKRHRAVRLIFTIMIVYFLFWAPYNIVLLLNTFQEFFGLNNCSSSNRLDQAMQVTETLGMTHCCINPIIYAFVGEKFRNYLLVFFQKHIAKRFCKCCSIFQQEAPERASSVYTRSTGEQEISVGL',
 'DMS': array([[   nan,    nan,    nan, ...,    nan,    nan,    nan],
        [ 0.35 , -0.12 ,    nan, ...,  0.04 , -0.24 , -0.53 ],
        [ 0.485,  0.15 , -0.015, ..., -0.065,  0.185,    nan],
        ...,
        [   nan,    nan,    nan, ...,    nan,    nan,    nan],
        [   nan,    nan,    nan, ...,    nan,    nan,    nan],
        [   nan,    nan,    nan, ...,    nan,    nan,    nan]],
       shape=(352, 20)),
 'log_probs': array([[-3.1606166, -5.071048 , -3.0783718, ..., -3.385966 , -4.684581 ,
         -3.631747 ],
        [-2.6697323, -5.0672083, -2.2900174, ..., -3.2828686, -4.9012537,
         -3.4445126],
  

In [37]:
path = '/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/'
filename = 'pg2_ProGym_matrices.pickle' 

with open(path + filename, 'wb') as f:
    pickle.dump(pg_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# for k in pg_dict.keys():
    
#     sequence = pg_dict[k]['sequence']

#     if len(sequence) <= 1024:
#         lp, rlp, llr = collect_log_prob_pg2(sequence, model, tokenizer)

#         pg_dict[k]['log_probs'] = lp
#         pg_dict[k]['ref_log_probs'] = rlp
#         pg_dict[k]['llr_matrix'] = llr
#     else:
#         pg_dict[k]['log_probs'] = None
#         pg_dict[k]['ref_log_probs'] = None
#         pg_dict[k]['llr_matrix'] = None

In [ ]:
# M_list = []

# for filename in os.listdir(directory):
#     m_list=[]
#     if filename.lower().endswith(".csv"):

#         filename_list = filename.rsplit(".", 1)[0]
#         parts = filename_list.split("_")

#         if len(parts) >= 2:
#             name = "_".join(parts[:2])

#         filepath = os.path.join(directory, filename)

#         with open(filepath, newline="", encoding="utf-8") as f:
#             reader = csv.reader(f)
#             for row in reader:
#                 if row:  # make sure row is not empty
#                     m_list.append(row[0])
#         t = tuple([name,m_list[1:]])
#         M_list.append(t)                
                

In [14]:
path = '/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/'
filename = 'pg2_wild_types_matrices.pickle' 

with open(path + filename, 'wb') as f:
    pickle.dump(pg_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

In [3]:
df_otc_h = pd.read_csv('/Users/johnhutchens/Desktop/Practicum/Data/DMS_ProteinGym_substitutions/OTC_HUMAN_Lo_2023.csv')

In [4]:
df_otc_h.head()

,mutant,mutated_sequence,DMS_score,DMS_score_bin
0,A102E,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.438,1
1,A102G,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.676,1
2,A102P,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.153,0
3,A102S,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.948,1
4,A102T,MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLK...,0.861,1


In [16]:
mut = df_otc_h.iloc[0]['mutant']
seq = df_otc_h.iloc[0]['mutated_sequence']
print(mut)
print(seq)

A102E
MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLKNFTGEEIKYMLWLSADLKFRIKQKGEYLPLLQGKSLGMIFEKRSTRTRLSTETGFELLGGHPCFLTTQDIHLGVNESLTDTARVLSSMADAVLARVYKQSDLDTLAKEASIPIINGLSDLYHPIQILADYLTLQEHYSSLKGLTLSWIGDGNNILHSIMMSAAKFGMHLQAATPKGYEPDASVTKLAEQYAKENGTKLLLTNDPLEAAHGGNVLITDTWISMGQEEEKKKRLQAFQGYQVTMKTAKVAASDWTFLHCLPRKPEEVDDEVFYSPRSLVFPEAENRKWTIMAVMVSLLTDYSPQLQKPKF


In [34]:
len_mut = len(mut)
orig = mut[0]
pos = int(mut[1:len_mut-1])-1
new = mut[len_mut-1]

In [35]:
wild_seq = seq[:pos] + orig + seq[pos+1:]

In [38]:
print(seq[101])
wild_seq[101]

E


'A'

In [23]:
with open('/Users/johnhutchens/Desktop/Practicum/Data/OTC_Human/pg2_otc_hum_matrices.pickle',
           'rb') as f:
    pg_dict = pickle.load(f)

In [39]:
pg_dict[None] = {'mutated_sequence': wild_seq}

In [40]:
pg_dict[None]

{'mutated_sequence': 'MLFNLRILLNNAAFRNGHNFMVRNFRCGQPLQNKVQLKGRDLLTLKNFTGEEIKYMLWLSADLKFRIKQKGEYLPLLQGKSLGMIFEKRSTRTRLSTETGFALLGGHPCFLTTQDIHLGVNESLTDTARVLSSMADAVLARVYKQSDLDTLAKEASIPIINGLSDLYHPIQILADYLTLQEHYSSLKGLTLSWIGDGNNILHSIMMSAAKFGMHLQAATPKGYEPDASVTKLAEQYAKENGTKLLLTNDPLEAAHGGNVLITDTWISMGQEEEKKKRLQAFQGYQVTMKTAKVAASDWTFLHCLPRKPEEVDDEVFYSPRSLVFPEAENRKWTIMAVMVSLLTDYSPQLQKPKF'}